####Change Data Feed (CDF) 

![](/Workspace/Users/ayush.gunjal@hginsights.com/Demo/images/cdc new.png)

[Reference Pipeline Logs- Source To Landingzone](https://dbc-ce68f8dc-9123.cloud.databricks.com/jobs/398755772020218/runs/258407977329897?o=7474660934313908)

[Reference Pipeline Logs- Landing To Bronze](https://dbc-ce68f8dc-9123.cloud.databricks.com/jobs/848602825634957/runs/233057187876526?o=7474660934313908)

[Reference Pipeline Logs- Bronze To Silver](https://dbc-ce68f8dc-9123.cloud.databricks.com/jobs/549822323606496/runs/643827535236975?o=7474660934313908)

In [0]:
%python

from pyspark.sql import functions as F

tables = [
    "crm_accounts", "crm_campaign_members", "crm_campaigns",
    "crm_contacts", "crm_opportunities", "crm_tasks",
    "crm_users", "events_raw"
]

schemas = spark.sql("SHOW SCHEMAS IN bronze").collect()
cust_schemas = sorted([s["databaseName"] for s in schemas if s["databaseName"].startswith("cust_")])

print("=" * 70)
print("  CDF Status: All bronze.cust_*.* tables")
print("=" * 70)

for schema in cust_schemas:
    print(f"\n  --- {schema} ---")
    for t in tables:
        full_name = f"bronze.{schema}.{t}"
        try:
            props = spark.sql(f"SHOW TBLPROPERTIES {full_name}").collect()
            cdf_prop = [r for r in props if r["key"] == "delta.enableChangeDataFeed"]
            if cdf_prop and cdf_prop[0]["value"].lower() == "true":
                print(f"    ✅ {t} - CDF ENABLED")
            else:
                print(f"    ❌ {t} — CDF NOT ENABLED")
        except Exception:
            print(f"    ⚠️  {t} — table not found")
